Imports

In [1]:
import pandas as pd
import numpy as np
import time

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, recall_score

Load the cleaned dataset

In [2]:
df = pd.read_pickle("../data/cleaned/NSDUH_2023_clean.pkl")

TARGET = "IRAMDEYR"
WEIGHT = "ANALWT2_C"   # we drop it immediately

y = df[TARGET]
X = df.drop(columns=[TARGET, WEIGHT])

print("X:", X.shape, " y:", y.shape)

X: (45133, 2315)  y: (45133,)


In [3]:
# Train Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train:", X_train.shape, " Test:", X_test.shape)

Train: (36106, 2315)  Test: (9027, 2315)


ElasticNet model (AUC version)

In [4]:
elasticnet_auc = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(
        penalty='elasticnet',
        solver='saga',
        C=1.0,
        l1_ratio=0.5,
        max_iter=500,
        random_state=42
    ))
])

print("\nRunning ElasticNet (AUC)")
start = time.time()

elasticnet_auc.fit(X_train, y_train)

print(f"Completed in {(time.time() - start)/60:.2f} minutes")


Running ElasticNet (AUC)
Completed in 15.58 minutes


c:\Users\agila\Documents\STM\Predicting-Past-Year-Major-Depressive-Episode\venv\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Evaluate on test set (AUC)

In [5]:
probs = elasticnet_auc.predict_proba(X_test)[:, 1]
auc_score = roc_auc_score(y_test, probs)

print("AUC Score:", auc_score)

AUC Score: 0.9990896286703606


Extract Top 100 features (AUC)

In [6]:
coefs = np.abs(elasticnet_auc.named_steps['clf'].coef_[0])
importances = pd.Series(coefs, index=X.columns)

top100_auc = importances.sort_values(ascending=False).head(100).index.tolist()

pd.Series(top100_auc).to_csv("../results/elasticnet_auc_top100.csv", index=False)

print("Saved elasticnet_auc_top100.csv")

Saved elasticnet_auc_top100.csv


ElasticNet model (Recall version)

In [7]:
elasticnet_recall = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(
        penalty='elasticnet',
        solver='saga',
        C=0.5,
        l1_ratio=0.5,
        max_iter=500,
        random_state=42
    ))
])

print("\nRunning ElasticNet (Recall)")
start = time.time()

elasticnet_recall.fit(X_train, y_train)

print(f"Completed in {(time.time() - start)/60:.2f} minutes")


Running ElasticNet (Recall)
Completed in 16.57 minutes


c:\Users\agila\Documents\STM\Predicting-Past-Year-Major-Depressive-Episode\venv\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Evaluate Recall

In [8]:
pred = elasticnet_recall.predict(X_test)
rec = recall_score(y_test, pred)

print("Recall Score:", rec)

Recall Score: 0.9537126325940212


Extract Top 100 Recall features

In [9]:
coefs_recall = np.abs(elasticnet_recall.named_steps['clf'].coef_[0])
importances_recall = pd.Series(coefs_recall, index=X.columns)

top100_recall = importances_recall.sort_values(ascending=False).head(100).index.tolist()

pd.Series(top100_recall).to_csv("../results/elasticnet_recall_top100.csv", index=False)

print("Saved elasticnet_recall_top100.csv")

Saved elasticnet_recall_top100.csv
